|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Reading through the table<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: write the paged attention oracle<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn.functional as F

import cudalib
from tests.helpers import build_paged, random_kv

Write the reference implementation of paged attention.

This is stage 07. Speed is not the point. This function is the **oracle**. You
check every later kernel against it.

It must stay correct on a shuffled pool. It must stay correct on [ragged](../../GLOSSARY.md#flat-batch)
lengths. It must stay correct when the unused slots hold another request's
tokens.

Make it correct, and accept that it is slow. Stage 08 recovers the speed.

In [ ]:
### run this cell

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_seqs, num_heads, num_kv_heads, head_dim, context_len = 4, 8, 2, 64, 100
BLOCK_SIZE = 16

keys, values = random_kv(num_seqs, num_kv_heads, context_len, head_dim, device)
query = torch.randn(num_seqs, num_heads, head_dim, device=device)
key_cache, value_cache, block_tables, context_lens = build_paged(keys, values, BLOCK_SIZE)

print(f'pool {tuple(key_cache.shape)}, block table {tuple(block_tables.shape)}')
print(f'sequence 0 lives in blocks {block_tables[0].tolist()}')

# Exercise 1: the dense oracle first

Use no paging. Compute attention over the original contiguous tensors. This
gives you something to check the paged version against.

In [ ]:
def reference_attention(query, keys, values, scale=None):
  num_seqs, num_heads, head_dim = query.shape
  group = num_heads // keys.shape[1]
  scale = scale or 1.0/math.sqrt(head_dim)
  output = torch.empty_like(query)
  for seq in range(num_seqs):
    for head in range(num_heads):
      scores = (keys[seq, head//group] @ query[seq,head]) * scale
      output[seq,head] = torch.softmax(scores, dim=0) @ values[seq, head//group]
  return output

expected = reference_attention(query, keys, values)
print('oracle:', tuple(expected.shape))

# Exercise 2: the scatter

Attention cannot read the pool until something writes it. You receive one flat
slot per token. Put K and V in the correct places.

In [ ]:
def write_kv(key_cache, value_cache, key, value, slot_indices):
  """key/value (T, KVH, D), slot_indices (T,) flat slots."""
  block_size = key_cache.shape[2]
  for token, slot in enumerate(slot_indices.tolist()):
    block, offset = slot // block_size, slot % block_size
    key_cache[block, :, offset]   = key[token]
    value_cache[block, :, offset] = value[token]

written_keys = torch.zeros_like(key_cache)
written_values = torch.zeros_like(value_cache)
slots = torch.tensor([block_tables[0,0]*BLOCK_SIZE + 0, block_tables[0,0]*BLOCK_SIZE + 1])
write_kv(written_keys, written_values, keys[0,:, :2].permute(1,0,2), values[0,:, :2].permute(1,0,2), slots)
print('wrote 2 tokens; matches source:',
      torch.allclose(written_keys[block_tables[0,0], :, :2], keys[0,:, :2]))

# Exercise 3: the gather

Walk the block table. Collect the blocks. Cut the result to `context_len`.
Then do the arithmetic from Exercise 1.

In [ ]:
def paged_attention(query, key_cache, value_cache, block_tables, context_lens, scale=None):
  num_seqs, num_heads, head_dim = query.shape
  num_kv_heads, block_size = key_cache.shape[1], key_cache.shape[2]
  group = num_heads // num_kv_heads
  scale = scale or 1.0/math.sqrt(head_dim)
  output = torch.empty_like(query)

  for seq in range(num_seqs):
    context_len = int(context_lens[seq])
    blocks = block_tables[seq, :(context_len + block_size - 1)//block_size].long()
    seq_keys = key_cache[blocks].permute(1,0,2,3).reshape(num_kv_heads, -1, head_dim)[:, :context_len]
    seq_values = value_cache[blocks].permute(1,0,2,3).reshape(num_kv_heads, -1, head_dim)[:, :context_len]
    for head in range(num_heads):
      kv_head = head // group
      scores = (seq_keys[kv_head] @ query[seq,head]) * scale
      output[seq,head] = torch.softmax(scores, dim=0) @ seq_values[kv_head]
  return output

output = paged_attention(query, key_cache, value_cache, block_tables, context_lens)
print('max difference from the oracle:', (output - expected).abs().max().item())

# Exercise 4: two quiet failures

The allocator recycles blocks, so the slots past `context_len` hold another
request's tokens. And no two sequences have the same length.

In [ ]:
poisoned_keys, poisoned_values = key_cache.clone(), value_cache.clone()
for seq in range(num_seqs):
  for block in range(block_tables.shape[1]):
    block_id = int(block_tables[seq,block])
    for offset in range(BLOCK_SIZE):
      if block*BLOCK_SIZE + offset >= context_len:
        poisoned_keys[block_id,:,offset] = 999.0
        poisoned_values[block_id,:,offset] = 999.0

poisoned_output = paged_attention(query, poisoned_keys, poisoned_values, block_tables, context_lens)
print('output unchanged:', torch.allclose(output, poisoned_output, atol=1e-5))

ragged = torch.tensor([100, 1, 17, 64], dtype=torch.int32, device=device)
ragged_output = paged_attention(query, key_cache, value_cache, block_tables, ragged)
print('ragged context lengths ran:', tuple(ragged_output.shape))

# Exercise 5: what did paging cost?

In [ ]:
if device == 'cuda':
  num_seqs, num_heads, num_kv_heads, head_dim, context_len = 32, 16, 8, 128, 512
  bench_keys, bench_values = random_kv(num_seqs, num_kv_heads, context_len, head_dim, device, dtype=torch.float16)
  bench_query = torch.randn(num_seqs, num_heads, head_dim, device=device, dtype=torch.float16)
  bench_key_cache, bench_value_cache, bench_block_tables, bench_context_lens = build_paged(bench_keys, bench_values, BLOCK_SIZE)

  paged = cudalib.bench_ms(lambda: paged_attention(bench_query, bench_key_cache, bench_value_cache, bench_block_tables, bench_context_lens),
                           iters=5, warmup=2)
  dense = cudalib.bench_ms(lambda: F.scaled_dot_product_attention(
            bench_query.unsqueeze(2), bench_keys.repeat_interleave(num_heads//num_kv_heads,1),
            bench_values.repeat_interleave(num_heads//num_kv_heads,1)), iters=20, warmup=5)
  print(f'contiguous SDPA: {dense:8.3f} ms')
  print(f'your paged loop: {paged:8.3f} ms   ({paged/dense:.0f}x slower)')

### You built the oracle, and you received the bill

Your `paged_attention` is correct on a shuffled pool, on ragged lengths, and
with recycled blocks. Keep it. Every kernel in the next three stages uses this
function as its only reference.

It also runs about fifteen times slower than the contiguous version. You wrote
both reasons yourself:

- **A Python loop over sequences.** One gather and one matmul per sequence per
  head. That is thousands of kernel launches where one belongs.
- **`key_cache[blocks]` materialises the data.** You allocate the whole gathered K
  and V, write them to memory, and read them back to compute a softmax. At 512
  tokens of context that is more traffic than the cache itself.

One kernel removes both problems. The second fix is the interesting one. The
score row never has to exist. Fold each tile into a running softmax instead.

    ./vc guide 8